
# Single‑Model Eval → Heatmap (Difficulty × Problem Type)

This notebook does three things:
1. **Load one model** (OpenAI by default — Anthropic/Mistral optional)  
2. **Run eval** over your dataset (with caching & error handling)  
3. **Plot a heatmap** of accuracy across *difficulty* × *problem type*

> It uses only **one** model — no multi‑model loops — and ignores/handles errored outputs so your analysis won't crash.


In [ ]:

# === 0) Config ===
# Set your dataset path and model/provider here.

DATA_PATH = "data/eval.csv"     # <-- change this to your dataset CSV
RESULTS_DIR = "results"         # where to store cached runs
PROVIDER = "openai"             # "openai", "anthropic", or "mistral"
MODEL = "gpt-4o-mini"           # e.g., OpenAI: "gpt-4o-mini", "o3-mini"; Anthropic: "claude-3-5-sonnet-latest"; Mistral: "mistral-large-latest"

SYSTEM_PROMPT = "You are a careful assistant. Solve the problem and return only the final answer."
TEMPERATURE = 0.0
TOP_P = 1.0

# Eval options
MAX_ROWS = None   # set e.g. 100 for a quick pass; None for all rows
CACHE_EVERY = 10  # save partial results every N calls

# Environment variables expected:
# - OPENAI_API_KEY          (for PROVIDER == "openai")
# - ANTHROPIC_API_KEY       (for PROVIDER == "anthropic")
# - MISTRAL_API_KEY         (for PROVIDER == "mistral")


In [ ]:

# === 1) Imports ===
import os, re, math, time, json, hashlib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make folders
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

# === 2) Utilities: column detection, answer extraction, correctness, caching ===

def detect_column(df: pd.DataFrame, candidates, default=None):
    cols = [c for c in candidates if c in df.columns]
    return cols[0] if cols else default

# -------- Robust model answer extraction --------
import json as _json
import re as _re

def _strip_code_fences(text: str) -> str:
    if text is None:
        return ""
    text = text.strip()
    fence = _re.compile(r"^```[a-zA-Z0-9_-]*\s*([\s\S]*?)\s*```$", _re.MULTILINE)
    m = fence.match(text)
    if m:
        return m.group(1).strip()
    return text

def _maybe_load_json(text: str):
    if not isinstance(text, str):
        return None
    try:
        return _json.loads(_strip_code_fences(text))
    except Exception:
        return None

def _collect_text_candidates_from_dict(d: dict):
    cands = []
    # OpenAI Chat Completions
    try:
        cands.append(d["choices"][0]["message"]["content"])
    except Exception:
        pass
    # OpenAI Responses API
    try:
        cands.append(d["output_text"])
    except Exception:
        pass
    try:
        out = d.get("output", [])
        if isinstance(out, list):
            for x in out:
                if isinstance(x, dict):
                    c = x.get("content", "")
                    if isinstance(c, str):
                        cands.append(c)
    except Exception:
        pass
    # Anthropic
    try:
        content = d.get("content")
        if isinstance(content, list):
            for b in content:
                if isinstance(b, dict) and b.get("type") == "text" and isinstance(b.get("text"), str):
                    cands.append(b["text"])
        if isinstance(d.get("completion"), str):
            cands.append(d["completion"])
    except Exception:
        pass
    # Generic keys
    for key in ["answer", "final", "final_answer", "result", "response", "text"]:
        v = d.get(key)
        if isinstance(v, str):
            cands.append(v)
    # Tool call arguments
    try:
        tool_calls = d["choices"][0]["message"].get("tool_calls", [])
        for t in tool_calls:
            fargs = t.get("function", {}).get("arguments", "")
            loaded = _maybe_load_json(fargs)
            if isinstance(loaded, dict):
                for k in ["answer", "final", "response", "text"]:
                    vv = loaded.get(k)
                    if isinstance(vv, str):
                        cands.append(vv)
    except Exception:
        pass

    # Deduplicate
    seen = set(); result = []
    for x in cands:
        if isinstance(x, str):
            xx = x.strip()
            if xx and xx not in seen:
                result.append(xx); seen.add(xx)
    return result

def extract_model_answer(payload):
    candidates = []
    if isinstance(payload, str):
        loaded = _maybe_load_json(payload)
        if isinstance(loaded, dict):
            candidates.extend(_collect_text_candidates_from_dict(loaded))
        else:
            candidates.append(payload.strip())
    elif isinstance(payload, dict):
        candidates.extend(_collect_text_candidates_from_dict(payload))
    elif isinstance(payload, list):
        for item in payload:
            if isinstance(item, (dict, list, str)):
                ans = extract_model_answer(item)
                if ans:
                    candidates.append(ans)
    # Heuristic: Final Answer marker
    expanded = []
    for c in candidates:
        parts = _re.split(r"(?i)final\s*answer\s*:\s*", c)
        expanded.append(parts[-1].strip() if len(parts) > 1 else c.strip())
    cleaned = [_re.sub(r"^(assistant|model)\s*:\s*", "", _strip_code_fences(x)).strip() for x in expanded if isinstance(x, str)]
    for cand in reversed(cleaned):
        if cand:
            return cand
    return ""

# -------- Correctness helpers --------
_num_re = re.compile(r"^[\+\-]?(?:\d+\.?\d*|\.\d+)(?:[eE][\+\-]?\d+)?$")

def _strip_latex(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)
    s = re.sub(r"\\\((.+?)\\\)", r"\1", s)
    s = re.sub(r"\\\[(.+?)\\\]", r"\1", s)
    s = s.replace("$", "")
    return s.strip()

def _as_number(s: str):
    if s is None:
        return None
    s = _strip_latex(str(s)).strip().replace(",", "")
    return float(s) if _num_re.match(s) else None

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = _strip_latex(s)
    s = re.sub(r"\\,","", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def answers_match(gold, pred, rel=1e-6, abs_=1e-9) -> bool:
    gnum, pnum = _as_number(gold), _as_number(pred)
    if gnum is not None and pnum is not None:
        return math.isclose(gnum, pnum, rel_tol=rel, abs_tol=abs_)
    return normalize_text(pred) == normalize_text(gold)

def ensure_eval_columns(df: pd.DataFrame) -> (pd.DataFrame, str, str):
    out = df.copy()
    topic_col = detect_column(out, ['problem_type','topic','type','category','subject'], default=None)
    if topic_col is None:
        out['problem_type'] = 'Unknown'
        topic_col = 'problem_type'
    diff_col = detect_column(out, ['difficulty','level','tier'], default=None)
    if diff_col is None:
        out['difficulty'] = 'Unknown'
        diff_col = 'difficulty'
    if 'is_correct' not in out.columns:
        gold_col = detect_column(out, ['answer','gold','expected','ground_truth','label'], default=None)
        pred_col = detect_column(out, ['model_answer','prediction','pred','model_output','response','answer_pred'], default=None)
        if gold_col is not None and pred_col is not None:
            out['is_correct'] = out.apply(lambda r: answers_match(r.get(gold_col,''), r.get(pred_col,'')), axis=1)
        else:
            out['is_correct'] = False
    return out, topic_col, diff_col

# -------- Caching helpers --------
def hash_row(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


In [ ]:

# === 3) Model client and generation (single provider) ===
# By default we only use the provider/model specified in Config.

def openai_generate(prompt: str, model: str, system: str = "", temperature: float = 0.0, top_p: float = 1.0):
    try:
        from openai import OpenAI
        client = OpenAI()
        # Prefer Responses API; fall back to Chat Completions if needed.
        try:
            resp = client.responses.create(
                model=model,
                input=[
                    {"role":"system","content":system or ""},
                    {"role":"user","content":prompt}
                ],
                temperature=temperature,
                top_p=top_p,
            )
            # Convert to dict-ish for extractor
            payload = resp.model_dump() if hasattr(resp, "model_dump") else resp
            return extract_model_answer(payload), payload, None
        except Exception as e1:
            # Fallback to Chat Completions
            try:
                cc = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role":"system","content":system or ""},
                        {"role":"user","content":prompt}
                    ],
                    temperature=temperature,
                    top_p=top_p,
                )
                payload = cc.model_dump() if hasattr(cc, "model_dump") else cc
                return extract_model_answer(payload), payload, None
            except Exception as e2:
                return "", None, f"OpenAIError: {e2}"
    except Exception as e:
        return "", None, f"OpenAIClientError: {e}"

def anthropic_generate(prompt: str, model: str, system: str = "", temperature: float = 0.0, top_p: float = 1.0):
    try:
        import anthropic
        client = anthropic.Anthropic()
        msg = client.messages.create(
            model=model,
            max_tokens=2048,
            system=system or "",
            messages=[{"role":"user", "content": prompt}],
            temperature=temperature,
            top_p=top_p
        )
        payload = msg.__dict__ if hasattr(msg, "__dict__") else msg
        return extract_model_answer(payload), payload, None
    except Exception as e:
        return "", None, f"AnthropicError: {e}"

def mistral_generate(prompt: str, model: str, system: str = "", temperature: float = 0.0, top_p: float = 1.0):
    try:
        from mistralai import Mistral
        client = Mistral()
        chat = client.chat.complete(
            model=model,
            messages=[
                {"role":"system","content":system or ""},
                {"role":"user","content":prompt}
            ],
            temperature=temperature,
            top_p=top_p
        )
        payload = chat.__dict__ if hasattr(chat, "__dict__") else chat
        return extract_model_answer(payload), payload, None
    except Exception as e:
        return "", None, f"MistralError: {e}"

def generate_answer(prompt: str, model: str, provider: str, system: str = "", temperature: float = 0.0, top_p: float = 1.0):
    provider = (provider or "").lower().strip()
    if provider == "openai":
        return openai_generate(prompt, model, system, temperature, top_p)
    elif provider == "anthropic":
        return anthropic_generate(prompt, model, system, temperature, top_p)
    elif provider == "mistral":
        return mistral_generate(prompt, model, system, temperature, top_p)
    else:
        return "", None, f"Unknown provider: {provider}"


In [ ]:

# === 4) Load data ===
from pathlib import Path

if not Path(DATA_PATH).exists():
    raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

# Detect input column (the question/prompt)
INPUT_COL = detect_column(df, ['question','prompt','input','problem','query','text'], default=None)
if INPUT_COL is None:
    raise RuntimeError("Could not detect an input column. Add one of ['question','prompt','input','problem','query','text'].")

# Detect gold answer column if present
GOLD_COL = detect_column(df, ['answer','gold','expected','ground_truth','label'], default=None)
print(f"Detected input column: {INPUT_COL}")
print(f"Detected gold/answer column: {GOLD_COL}")
print(f"Rows in dataset: {len(df)}")

if MAX_ROWS is not None:
    df = df.head(int(MAX_ROWS)).copy()
    print(f"Subsampled rows: {len(df)}")


In [ ]:

# === 5) Evaluate with caching and error handling (single model) ===
cache_path = Path(RESULTS_DIR) / f"eval_cache_{PROVIDER}_{MODEL.replace('/','-')}.csv"
if cache_path.exists():
    cache_df = pd.read_csv(cache_path)
    print(f"Loaded cache: {len(cache_df)} rows from {cache_path}")
else:
    cache_df = pd.DataFrame(columns=['hash','model','model_answer','raw','error'])

# Merge cache to current df by hash of input
df['_row_hash'] = df[INPUT_COL].astype(str).map(hash_row)
cache_map = dict(zip(cache_df['hash'], cache_df['model_answer']))
cache_err = dict(zip(cache_df['hash'], cache_df['error']))

to_eval_mask = ~df['_row_hash'].isin(cache_df['hash'])
num_to_eval = to_eval_mask.sum()
print(f"Rows to evaluate: {num_to_eval} (cached: {len(df) - num_to_eval})")

evaluated = 0
new_rows = []

for idx, row in df.iterrows():
    h = row['_row_hash']
    if h in cache_map or h in cache_err:
        continue  # already cached (answer or error)

    prompt = str(row[INPUT_COL])

    ans, raw, err = generate_answer(
        prompt=prompt,
        model=MODEL,
        provider=PROVIDER,
        system=SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    new_rows.append({
        'hash': h,
        'model': MODEL,
        'model_answer': ans,
        'raw': json.dumps(raw) if raw is not None else "",
        'error': err or ""
    })
    evaluated += 1

    if evaluated % CACHE_EVERY == 0:
        # append to cache on disk
        if new_rows:
            cache_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)
            cache_df.drop_duplicates(subset=['hash'], keep='last', inplace=True)
            cache_df.to_csv(cache_path, index=False)
            new_rows = []
        print(f"…evaluated {evaluated} rows; cache saved.")

# Flush remaining
if new_rows:
    cache_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)
    cache_df.drop_duplicates(subset=['hash'], keep='last', inplace=True)
    cache_df.to_csv(cache_path, index=False)

print(f"Done. Cache now has {len(cache_df)} rows.")


In [ ]:

# === 6) Assemble eval_df (join original + cache), compute correctness if possible ===
eval_df = df.copy()

# Join cache on hash
eval_df = eval_df.merge(cache_df[['hash','model','model_answer','error']], how='left', left_on='_row_hash', right_on='hash')

# If dataset already had predictions, we prefer the fresh cache unless absent
pred_col = 'model_answer'
if pred_col not in eval_df.columns:
    eval_df[pred_col] = eval_df['model_answer']

# Compute correctness if we have gold answers
eval_df_fixed, topic_col, diff_col = ensure_eval_columns(eval_df)

answered = eval_df_fixed[eval_df_fixed['error'].isna() | (eval_df_fixed['error'].astype(str) == "")]
errored  = eval_df_fixed[~(eval_df_fixed['error'].isna() | (eval_df_fixed['error'].astype(str) == ""))]

print(f"Answered rows: {len(answered)} | Errored rows: {len(errored)}")
if len(errored):
    print("Top error types:")
    print(errored['error'].fillna('').str.split(':').str[0].value_counts().head())

# Persist a results file per model
out_path = Path(RESULTS_DIR) / f"eval_results_{PROVIDER}_{MODEL.replace('/','-')}.csv"
eval_df_fixed.to_csv(out_path, index=False)
print(f"Saved eval results → {out_path}")

# Make a trimmed DataFrame used for plotting (answered only)
plot_df = answered.copy()


In [ ]:

# === 7) Heatmap: Difficulty × Problem Type (accuracy over answered rows) ===
if len(plot_df) == 0:
    raise RuntimeError("No answered rows available for plotting. Check API keys, provider/model, or dataset.")

# Build accuracy pivot
piv = plot_df.pivot_table(index=topic_col, columns=diff_col, values='is_correct', aggfunc='mean', fill_value=0.0)
piv = piv.sort_index().reindex(sorted(piv.columns), axis=1)

import numpy as np
fig, ax = plt.subplots(figsize=(max(6, len(piv.columns)*0.9), max(4, len(piv.index)*0.5)))
im = ax.imshow(piv.values, aspect='auto')  # default colormap

ax.set_xticks(np.arange(len(piv.columns)))
ax.set_yticks(np.arange(len(piv.index)))
ax.set_xticklabels(list(piv.columns), rotation=45, ha='right')
ax.set_yticklabels(list(piv.index))
ax.set_xlabel(diff_col.title())
ax.set_ylabel(topic_col.replace('_',' ').title())
ax.set_title(f"Accuracy Heatmap — {PROVIDER}:{MODEL} (answered rows only)")

# annotate
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f\"{piv.values[i, j]*100:.0f}%\", ha=\"center\", va=\"center\", fontsize=8)

fig.tight_layout()
plt.show()
